# Juan Diego Cadena, 329220

## 02 - Features y control de fuga

Las features nuevas incluyen rezagos de 364/365/371 días, calendario de la fecha objetivo, ciclos anuales, ventanas de feriado, el promedio de las cuatro semanas equivalentes y tendencia 7/28. Todas existen al cierre del día de ejecución. `obj_n_emergencias_t3/t7` solo construye la etiqueta y nunca entra a X.

In [1]:
from pathlib import Path
import os, sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import Image, display
from src.experimentos import *
df, data, info = cargar_contexto()
print(f'Fuente: {info.source}')
print(f'OBT: {info.rows:,} filas x {info.columns} columnas; corte {info.snapshot_end}')
print(f'Train: {len(data.train):,}; holdout: {len(data.test):,}; target positivo: {data.train[data.target_column].mean():.2%}')
print(f'Validación: 4 semestres completos; purga: {data.purge_days} días; features: {len(data.feature_columns)}')


Fuente: C:\Universidad\Data_Science\Data_science_end_to_end\laboratorios\semana-09-sesion-01\data\obt_emergencias_canton_dia.csv
OBT: 429,632 filas x 37 columnas; corte 2026-09-30
Train: 387,968; holdout: 40,320; target positivo: 8.17%
Validación: 4 semestres completos; purga: 3 días; features: 48


In [2]:
import pandas as pd
pd.DataFrame({
'feature':['media_mismo_dia_4','media_mismo_periodo_anio','ratio_estacional_p90','temporada_objetivo','dias_al_feriado_mas_cercano'],
'evidencia':['baseline estacional','mismo periodo del año anterior','niveles distintos entre cantones','Carnaval/fin de año/feriados','calendario oficial'],
'disponible_en_inferencia':['sí: lags 7/14/21/28','sí: lags 364/365/371','sí: P90 solo train','sí: fecha conocida','sí: calendario conocido'],
'codificacion':['numérica','numérica','numérica','categórica','numérica']})

,feature,evidencia,disponible_en_inferencia,codificacion
0,media_mismo_dia_4,baseline estacional,sí: lags 7/14/21/28,numérica
1,media_mismo_periodo_anio,mismo periodo del año anterior,sí: lags 364/365/371,numérica
2,ratio_estacional_p90,niveles distintos entre cantones,sí: P90 solo train,numérica
3,temporada_objetivo,Carnaval/fin de año/feriados,sí: fecha conocida,categórica
4,dias_al_feriado_mas_cercano,calendario oficial,sí: calendario conocido,numérica


In [3]:
forbidden={'obj_n_emergencias_t3','obj_n_emergencias_t7','fecha_objetivo'}
assert not forbidden.intersection(data.feature_columns)
assert data.train.fecha_objetivo.max() < data.test.fecha.min()
print('Sin columnas futuras, con purga t+3 y sin solapamiento temporal.')
data.train[['fecha','fecha_objetivo','cod_canton','lag_364d','temporada_objetivo','es_ventana_feriado_objetivo',data.target_column]].tail(12)

Sin columnas futuras, con purga t+3 y sin solapamiento temporal.


,fecha,fecha_objetivo,cod_canton,lag_364d,temporada_objetivo,es_ventana_feriado_objetivo,alta_demanda_t3
387956,2026-03-28,2026-03-31,2107,1.0,regular,False,0
387957,2026-03-28,2026-03-31,2201,50.0,regular,False,0
387958,2026-03-28,2026-03-31,2202,0.0,regular,False,0
387959,2026-03-28,2026-03-31,2203,19.0,regular,False,1
387960,2026-03-28,2026-03-31,2204,10.0,regular,False,0
387961,2026-03-28,2026-03-31,2301,269.0,regular,False,0
387962,2026-03-28,2026-03-31,2302,37.0,regular,False,0
387963,2026-03-28,2026-03-31,2401,46.0,regular,False,0
387964,2026-03-28,2026-03-31,2402,45.0,regular,False,0
387965,2026-03-28,2026-03-31,2403,32.0,regular,False,0


El percentil 90 se calcula exclusivamente con el conjunto de entrenamiento y se aplica después al holdout. La codificación ordinal dentro del `ColumnTransformer` evita una matriz one-hot muy grande y mantiene el experimento dentro de la memoria de la laptop.